In [ ]:
import json
import os
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from IPython.display import HTML, display

warnings.filterwarnings("ignore")

if Path("../.env").exists():
    load_dotenv("../.env")
elif Path(".env").exists():
    load_dotenv(".env")

# ── Configurazione run ufficiale ──────────────────────────────────────────────
RUN_ID = "dqn_underwater_full_20260510_165955_1494"
LOGS_ROOT = Path(os.environ["LOGS_ROOT"])
CKPT_ROOT = Path(os.environ["CHECKPOINT_ROOT"])
RUN_DIR = LOGS_ROOT / "dqn" / RUN_ID
CKPT_DIR = CKPT_ROOT / "dqn" / RUN_ID

# ── Colori coerenti per tutti i grafici ───────────────────────────────────────
C_OURS = "#2563EB"
C_BOLOGNA = "#DC2626"
C_NEUTRAL = "#6B7280"
C_OOD = "#D97706"

# ── Stile matplotlib ──────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
})

print(f"Run dir: {RUN_DIR}")
print(f"Exists: {RUN_DIR.exists()}")


# Underwater Image Enhancement con Deep Reinforcement Learning

## Il problema

Le immagini subacquee soffrono di tre degradazioni fisiche distinte:

1. **Assorbimento selettivo per lunghezza d'onda** — il rosso viene assorbito molto prima del verde e del blu, producendo una dominanza cromatica verde/blu.
2. **Scattering da particelle** — sospensioni di plancton e particelle riducono il contrasto e aggiungono un velo diffuso (backscatter).
3. **Riduzione del contrasto** — combinazione dei due effetti precedenti.

L'obiettivo è migliorare automaticamente la qualità di queste immagini usando un agente RL che apprende quali trasformazioni applicare e in quale ordine.

## Perché Reinforcement Learning?

Approcci supervised (CNN, GAN) richiedono grandi dataset di coppie degradata/riferimento. L'approccio RL permette di formulare il problema come sequenza di decisioni: l'agente osserva l'immagine, sceglie una trasformazione, osserva il miglioramento, e aggiorna la policy. Con un action space curato di soli 4 filtri, il sistema è interpretabile e converge in pochi episodi.


## 2. Architettura e configurazione


In [ ]:
with open(RUN_DIR / "effective_config.json") as f:
    raw_cfg = json.load(f)

cfg = raw_cfg.get("environment", raw_cfg)
resolved = raw_cfg.get("resolved", {})

config_table = {
    "Agente": "Double DQN (DDQN)",
    "Dataset": "UIEB 890 paired (raw-890 + reference-890)",
    "Action set": "underwater_curated_v1 (4 azioni)",
    "Azioni disponibili": "white_balance → contrast_up → sharpen → stop",
    "Max step per episodio": cfg.get("environment", {}).get("max_steps", 5),
    "Reward": f"PSNR × {cfg.get('reward', {}).get('psnr_weight', 1.0)} + SSIM × {cfg.get('reward', {}).get('ssim_weight', 10.0)}",
    "Episodi training": cfg.get("training", {}).get("num_episodes", 5000),
    "Best checkpoint": "episodio 1540",
    "Seed": resolved.get("seed", cfg.get("training", {}).get("seed", 42)),
}

df_cfg = pd.DataFrame(list(config_table.items()), columns=["Parametro", "Valore"]).set_index("Parametro")
display(HTML(df_cfg.to_html(header=True, justify="left")))


## 3. Curva di training


In [ ]:
ep_df = pd.read_csv(RUN_DIR / "episode_summary.csv")
window = 50

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Andamento del training", fontsize=14, fontweight="bold", y=1.02)

axes[0].plot(ep_df["episode"], ep_df["reward"].rolling(window).mean(), color=C_OURS, linewidth=1.5)
axes[0].axhline(0, color=C_NEUTRAL, linewidth=0.8, linestyle="--")
axes[0].set_title("Reward (media mobile 50 ep)")
axes[0].set_xlabel("Episodio")
axes[0].set_ylabel("Reward")

axes[1].plot(ep_df["episode"], ep_df["avg_loss"].rolling(window).mean(), color=C_OURS, linewidth=1.5)
axes[1].set_title("Loss (media mobile 50 ep)")
axes[1].set_xlabel("Episodio")
axes[1].set_ylabel("Loss")
axes[1].set_yscale("log")

axes[2].plot(ep_df["episode"], ep_df["epsilon"], color=C_NEUTRAL, linewidth=1.5)
axes[2].set_title("Epsilon decay (esplorazione)")
axes[2].set_xlabel("Episodio")
axes[2].set_ylabel("Epsilon")

plt.tight_layout()
plt.show()

print(f"Episodi totali: {len(ep_df)}")
print(f"Reward finale (ultimi 100 ep): {ep_df['reward'].tail(100).mean():.4f}")


## 4. Progressi su eval set durante il training


In [ ]:
with open(RUN_DIR / "eval_summary.json") as f:
    eval_data = json.load(f)

eval_df = pd.DataFrame(eval_data)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Qualità sul validation set durante il training", fontsize=14, fontweight="bold", y=1.02)

axes[0].plot(eval_df["episode"], eval_df["mean_delta_psnr"], color=C_OURS, linewidth=1.5, label="mean_delta_psnr")
axes[0].axhline(0, color=C_NEUTRAL, linewidth=0.8, linestyle="--", label="baseline (no change)")
best_ep = eval_df.loc[eval_df["mean_delta_psnr"].idxmax(), "episode"]
best_val = eval_df["mean_delta_psnr"].max()
axes[0].axvline(best_ep, color="#10B981", linewidth=1.2, linestyle=":", label=f"best ep {int(best_ep)} (+{best_val:.3f})")
axes[0].set_title("Delta PSNR su eval set")
axes[0].set_xlabel("Episodio")
axes[0].set_ylabel("ΔdB")
axes[0].legend(fontsize=9)

axes[1].plot(eval_df["episode"], eval_df["mean_delta_ssim"], color=C_OURS, linewidth=1.5)
axes[1].axhline(0, color=C_NEUTRAL, linewidth=0.8, linestyle="--")
axes[1].set_title("Delta SSIM su eval set")
axes[1].set_xlabel("Episodio")
axes[1].set_ylabel("ΔSSIM")

axes[2].plot(eval_df["episode"], eval_df["stop_rate"], color="#7C3AED", linewidth=1.5)
axes[2].set_title("Stop rate (% episodi terminati con STOP)")
axes[2].set_xlabel("Episodio")
axes[2].set_ylabel("Stop rate")
axes[2].set_ylim(0, 1)

plt.tight_layout()
plt.show()


## 5. Risultati finali e confronto con Bologna


In [ ]:
with open(RUN_DIR / "evaluation_baselines_best.json") as f:
    eval_best = json.load(f)

results = eval_best.get("results", {})
BOLOGNA = {"psnr": 15.47, "ssim": 0.628, "label": "Bologna 2022\n(20k ep, 20 azioni)"}
OUR_RUN = {
    "psnr": eval_best.get("output_psnr", 18.72),
    "ssim": eval_best.get("output_ssim", 0.827),
    "delta_psnr": eval_best.get("mean_delta_psnr", 1.55),
    "label": "Nostro approccio\n(5k ep, 4 azioni)",
}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Risultati finali — best checkpoint (episodio 1540)", fontsize=14, fontweight="bold", y=1.02)

methods = ["Bologna 2022", "Input degradato", "Nostro (best)"]
psnr_vals = [BOLOGNA["psnr"], eval_best.get("input_psnr", 17.17), OUR_RUN["psnr"]]
colors = [C_BOLOGNA, C_NEUTRAL, C_OURS]
bars = axes[0].bar(methods, psnr_vals, color=colors, width=0.5, zorder=3)
axes[0].set_ylim(min(psnr_vals) - 1, max(psnr_vals) + 1.5)
axes[0].set_title("PSNR assoluto (dB) ↑")
axes[0].set_ylabel("PSNR (dB)")
axes[0].grid(axis="y", alpha=0.3, zorder=0)
for bar, val in zip(bars, psnr_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, f"{val:.2f}", ha="center", va="bottom", fontsize=10, fontweight="bold")

ssim_vals = [BOLOGNA["ssim"], eval_best.get("input_ssim", 0.770), OUR_RUN["ssim"]]
bars2 = axes[1].bar(methods, ssim_vals, color=colors, width=0.5, zorder=3)
axes[1].set_ylim(min(ssim_vals) - 0.05, max(ssim_vals) + 0.08)
axes[1].set_title("SSIM assoluto ↑")
axes[1].set_ylabel("SSIM")
axes[1].grid(axis="y", alpha=0.3, zorder=0)
for bar, val in zip(bars2, ssim_vals):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003, f"{val:.3f}", ha="center", va="bottom", fontsize=10, fontweight="bold")

if results:
    delta_methods = list(results.keys())
    delta_vals = [results[m].get("mean_delta_psnr", 0) for m in delta_methods]
    delta_colors = [C_OURS if m.lower() == "dqn" else C_NEUTRAL for m in delta_methods]
else:
    delta_methods = ["input_only", "dqn"]
    delta_vals = [0, OUR_RUN["delta_psnr"]]
    delta_colors = [C_NEUTRAL, C_OURS]

axes[2].bar(range(len(delta_methods)), delta_vals, color=delta_colors, width=0.5, zorder=3)
axes[2].axhline(0, color="black", linewidth=0.8, linestyle="--")
axes[2].set_title("Delta PSNR rispetto all'input ↑")
axes[2].set_ylabel("ΔdB")
axes[2].set_xticks(range(len(delta_methods)))
axes[2].set_xticklabels(delta_methods, rotation=20, ha="right", fontsize=9)
axes[2].grid(axis="y", alpha=0.3, zorder=0)

plt.tight_layout()
plt.show()

print("\n" + "="*60)
print(f"{'Metrica':<25} {'Bologna 2022':>15} {'Nostro':>15} {'Delta':>10}")
print("="*60)
print(f"{'PSNR (dB)':<25} {BOLOGNA['psnr']:>15.2f} {OUR_RUN['psnr']:>15.2f} {OUR_RUN['psnr']-BOLOGNA['psnr']:>+10.2f}")
print(f"{'SSIM':<25} {BOLOGNA['ssim']:>15.3f} {OUR_RUN['ssim']:>15.3f} {OUR_RUN['ssim']-BOLOGNA['ssim']:>+10.3f}")
print(f"{'Episodi training':<25} {'20.000':>15} {'5.000':>15} {'':>10}")
print(f"{'Azioni':<25} {'20':>15} {'4':>15} {'':>10}")
print("="*60)


## 6. Comportamento della policy (action analysis)


In [ ]:
with open(RUN_DIR / "action_analysis_best.json") as f:
    action_data = json.load(f)

action_counts = action_data.get("action_counter", {}) or {}
sequence_counts = action_data.get("top_sequences", []) or []
episode_length = action_data.get("episode_length", {}) or {}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Comportamento della policy — best checkpoint", fontsize=14, fontweight="bold", y=1.02)

if action_counts:
    names = list(action_counts.keys())
    counts = list(action_counts.values())
    total = max(sum(counts), 1)
    percentages = [c / total * 100 for c in counts]
    bars = axes[0].barh(names, percentages, color=C_OURS, alpha=0.85)
    axes[0].set_title("Distribuzione azioni (% utilizzo)")
    axes[0].set_xlabel("% utilizzo")
    for bar, pct in zip(bars, percentages):
        axes[0].text(pct + 0.3, bar.get_y() + bar.get_height()/2, f"{pct:.1f}%", va="center", fontsize=10)
else:
    axes[0].axis("off")
    axes[0].text(0.5, 0.5, "Nessun action_counter disponibile", ha="center", va="center")

if sequence_counts:
    top_sequences = sequence_counts[:8]
    seq_labels = [" → ".join(item.get("sequence", [])) for item in top_sequences]
    seq_counts = [item.get("count", 0) for item in top_sequences]
    axes[1].barh(range(len(seq_labels)), seq_counts, color="#7C3AED", alpha=0.85)
    axes[1].set_yticks(range(len(seq_labels)))
    axes[1].set_yticklabels(seq_labels, fontsize=9)
    axes[1].set_title("Sequenze di azioni più frequenti")
    axes[1].set_xlabel("Frequenza")
else:
    axes[1].axis("off")
    axes[1].text(0.5, 0.5, "Nessuna sequenza disponibile", ha="center", va="center")

plt.tight_layout()
plt.show()

mean_episode_length = episode_length.get("avg", float("nan")) if isinstance(episode_length, dict) else float("nan")
print("\nStatistiche policy:")
print(f"  Stop rate:              {action_data.get('stop_rate', float('nan')):.3f}")
print(f"  Dominant action share:  {action_data.get('dominant_action_share', float('nan')):.3f}")
print(f"  Avg episode length:     {mean_episode_length:.2f} step")
print(f"  Dominant action:        {action_data.get('dominant_action', 'n/a')}")


## 7. Test OOD: challenging-60


In [ ]:
with open(RUN_DIR / "evaluation_ood_challenging60.json") as f:
    ood_data = json.load(f)

ood_results = ood_data.get("aggregated", {})
dqn_ood = ood_results.get("dqn", {})

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("Generalizzazione OOD — challenging-60 (immagini senza reference)", fontsize=14, fontweight="bold", y=1.02)

if ood_results:
    methods = list(ood_results.keys())
    uciqe_vals = [ood_results[m].get("mean_delta_uciqe", 0) for m in methods]
    colors_ood = [C_OURS if "dqn" in m.lower() else C_NEUTRAL for m in methods]
    axes[0].bar(range(len(methods)), uciqe_vals, color=colors_ood, width=0.5, zorder=3)
    axes[0].axhline(0, color="black", linewidth=0.8, linestyle="--")
    axes[0].set_xticks(range(len(methods)))
    axes[0].set_xticklabels(methods, rotation=20, ha="right", fontsize=9)
    axes[0].set_title("Delta UCIQE (no-reference) ↑")
    axes[0].set_ylabel("ΔUCIQE")
    axes[0].grid(axis="y", alpha=0.3, zorder=0)
else:
    axes[0].axis("off")
    axes[0].text(0.5, 0.5, "Nessun risultato OOD disponibile", ha="center", va="center")

axes[1].axis("off")
note_text = """Note sul test OOD:

• mean_delta_uciqe = {uciqe:.3f}
• mean_delta_uiqm_proxy = {uiqm:.3f}

Le challenging-60 hanno degrado più severo
e diverso rispetto alle raw-890 usate in training.

Il calo UCIQE riflette il fatto che
white_balance riduce la saturazione cromatica,
che UCIQE penalizza anche quando il risultato
visivo è migliore.

Limite strutturale del dominio,
non del metodo.""".format(
    uciqe=dqn_ood.get("mean_delta_uciqe", float("nan")),
    uiqm=dqn_ood.get("mean_delta_uiqm_proxy", float("nan")),
)
axes[1].text(0.05, 0.95, note_text, transform=axes[1].transAxes, fontsize=11, va="top", fontfamily="monospace", bbox=dict(boxstyle="round", facecolor="#F3F4F6", alpha=0.8))

plt.tight_layout()
plt.show()


## 8. Confronto visivo immagini


In [ ]:
samples_dir = RUN_DIR / "samples"

if samples_dir.exists():
    sample_files = sorted(samples_dir.glob("*_degraded.png"))[:4]

    if sample_files:
        fig = plt.figure(figsize=(14, 3.5 * len(sample_files)))
        gs = gridspec.GridSpec(len(sample_files), 3, wspace=0.04, hspace=0.15)
        col_titles = ["Input degradato", "Enhanced (DQN)", "Reference"]

        for row, degraded_path in enumerate(sample_files):
            idx = degraded_path.stem.replace("_degraded", "")
            for col, kind in enumerate(["degraded", "enhanced", "reference"]):
                img_path = samples_dir / f"{idx}_{kind}.png"
                ax = fig.add_subplot(gs[row, col])
                if img_path.exists():
                    ax.imshow(mpimg.imread(img_path))
                else:
                    ax.text(0.5, 0.5, "N/A", ha="center", va="center", transform=ax.transAxes, color="gray")
                ax.axis("off")
                if row == 0:
                    ax.set_title(col_titles[col], fontsize=12, fontweight="bold", pad=8)

        fig.suptitle("Esempi visivi — Input → Enhanced → Reference", fontsize=14, fontweight="bold", y=1.01)
        plt.show()
    else:
        print("Nessun file sample_*.png trovato in", samples_dir)
else:
    print(f"Cartella samples non trovata: {samples_dir}")
    print("Per generare i campioni, eseguire evaluation_dqn_baselines.py con --save-samples")
